In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


I’ll use a representative set spanning the generations, including 20 agents:

1. Cefalexin
2. Cefadroxil
3. Cefradine
4. Cefazolin
5. Cefuroxime
6. Cefaclor
7. Cefprozil
8. Cefoxitin
9. Cefotetan
10. Cefotiam
11. Cefixime
12. Cefpodoxime
13. Cefdinir
14. Ceftriaxone
15. Cefotaxime
16. Ceftazidime
17. Cefoperazone
18. Cefepime
19. Ceftaroline
20. Ceftobiprole

The revised workflow will also avoid assuming that 10 MIC observations represent 10 distinct compounds. Those are different sample-size measures, and the code will report them separately.

It seems like I can’t do more advanced data analysis right now. Please try again later.

I was unable to invoke the file-generation tool, so I couldn't create the revised ZIP project. The 20-compound list and revised workflow are defined above, but I can't provide a working download until file generation is available.

In [ ]:
from pathlib import Path
import textwrap, json, zipfile, os, py_compile

root = Path("/mnt/data/cephalosporin_20_pipeline")
(root / "data").mkdir(parents=True, exist_ok=True)
(root / "R").mkdir(exist_ok=True)
(root / "notebooks").mkdir(exist_ok=True)

files = {}

files["data/cephalosporins_20.csv"] = """compound,common_name,pubmed_name,generation
1,Cefalexin,cephalexin,1
2,Cefadroxil,cefadroxil,1
3,Cefradine,cephradine,1
4,Cefazolin,cefazolin,1
5,Cefuroxime,cefuroxime,2
6,Cefaclor,cefaclor,2
7,Cefprozil,cefprozil,2
8,Cefoxitin,cefoxitin,2
9,Cefotetan,cefotetan,2
10,Cefotiam,cefotiam,2
11,Cefixime,cefixime,3
12,Cefpodoxime,cefpodoxime,3
13,Cefdinir,cefdinir,3
14,Ceftriaxone,ceftriaxone,3
15,Cefotaxime,cefotaxime,3
16,Ceftazidime,ceftazidime,3
17,Cefoperazone,cefoperazone,3
18,Cefepime,cefepime,4
19,Ceftaroline,ceftaroline,5
20,Ceftobiprole,ceftobiprole,5
"""

files["requirements.txt"] = """requests>=2.31
pandas>=2.1
python-dotenv>=1.0
tqdm>=4.66
biopython>=1.83
openai>=1.30
ollama>=0.3
rdkit>=2023.9
scikit-learn>=1.4
matplotlib>=3.8
jupyter>=1.0
"""

files[".env.example"] = """# Choose one extraction backend: ollama or openai
LLM_PROVIDER=ollama
OLLAMA_MODEL=llama3.1
OPENAI_API_KEY=
OPENAI_MODEL=gpt-4.1-mini

# NCBI asks users to identify themselves; use your email.
NCBI_EMAIL=your.email@example.com
NCBI_API_KEY=
"""

files["cephalo_pipeline.py"] = r'''
"""
Literature retrieval and MIC extraction starter pipeline for 20 cephalosporins.

Important:
- PubMed search retrieves bibliographic records and abstracts, not necessarily MIC tables.
- Europe PMC full text is attempted only for articles with a PMCID and an available OA full-text XML.
- LLM-extracted MIC values are candidates and MUST be checked against the source article.
"""
from __future__ import annotations
import os, re, json, time, argparse
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
EUROPEPMC = "https://www.ebi.ac.uk/europepmc/webservices/rest"
ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

COMPOUNDS = pd.read_csv(DATA / "cephalosporins_20.csv")

def ncbi_params():
    p = {"tool": "cephalosporin_mic_literature_pipeline"}
    email = os.getenv("NCBI_EMAIL")
    api_key = os.getenv("NCBI_API_KEY")
    if email: p["email"] = email
    if api_key: p["api_key"] = api_key
    return p

def search_pubmed(term: str, retmax: int = 200) -> list[str]:
    params = {**ncbi_params(), "db":"pubmed", "term":term, "retmax":retmax, "retmode":"json", "sort":"relevance"}
    r = requests.get(BASE + "esearch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])

def fetch_pubmed_xml(pmids: list[str]) -> str:
    if not pmids: return ""
    params = {**ncbi_params(), "db":"pubmed", "id":",".join(pmids), "retmode":"xml"}
    r = requests.get(BASE + "efetch.fcgi", params=params, timeout=120)
    r.raise_for_status()
    return r.text

def parse_pubmed_xml(xml_text: str) -> pd.DataFrame:
    from Bio import Entrez
    from io import StringIO
    records = Entrez.read(StringIO(xml_text))
    rows = []
    for rec in records.get("PubmedArticle", []):
        med = rec.get("MedlineCitation", {})
        art = med.get("Article", {})
        pmid = str(med.get("PMID", ""))
        title = str(art.get("ArticleTitle", ""))
        abstract_parts = []
        for item in art.get("Abstract", {}).get("AbstractText", []):
            label = item.attributes.get("Label", "") if hasattr(item, "attributes") else ""
            abstract_parts.append((f"{label}: " if label else "") + str(item))
        authors = []
        for au in art.get("AuthorList", []):
            if au.get("LastName") and au.get("Initials"):
                authors.append(f'{au["LastName"]} {au["Initials"]}')
        journal = str(art.get("Journal", {}).get("Title", ""))
        year = ""
        try:
            year = str(art["Journal"]["JournalIssue"]["PubDate"].get("Year", ""))
        except Exception:
            pass
        pmcid = ""
        for aid in rec.get("PubmedData", {}).get("ArticleIdList", []):
            if getattr(aid, "attributes", {}).get("IdType") == "pmc":
                pmcid = str(aid)
        rows.append({
            "pmid": pmid, "pmcid": pmcid, "title": title,
            "abstract": " ".join(abstract_parts), "authors": "; ".join(authors),
            "journal": journal, "year": year,
            "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        })
    return pd.DataFrame(rows)

def retrieve_literature(retmax: int = 200, pause: float = 0.34) -> pd.DataFrame:
    all_ids = set()
    for row in COMPOUNDS.itertuples(index=False):
        name = row.pubmed_name
        # Search antimicrobial activity and susceptibility terms; broad enough to capture MIC papers.
        query = f'("{name}"[Title/Abstract]) AND (antimicrobial OR antibacterial OR susceptibility OR MIC OR "minimum inhibitory concentration")'
        ids = search_pubmed(query, retmax=retmax)
        all_ids.update(ids)
        time.sleep(pause)
        print(f"{name}: {len(ids)} PubMed records")
    ids = sorted(all_ids)
    print(f"Unique PMIDs: {len(ids)}")
    records = []
    for start in range(0, len(ids), 150):
        xml = fetch_pubmed_xml(ids[start:start+150])
        if xml:
            records.append(parse_pubmed_xml(xml))
        time.sleep(pause)
    df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    if not df.empty:
        df.to_csv(OUT / "pubmed_records.csv", index=False)
    return df

def fetch_europe_pmc_fulltext(pmcid: str) -> str:
    if not pmcid: return ""
    pmcid = pmcid if pmcid.startswith("PMC") else "PMC" + pmcid
    url = f"{EUROPEPMC}/{pmcid}/fullTextXML"
    r = requests.get(url, timeout=60)
    if r.status_code != 200: return ""
    return re.sub(r"<[^>]+>", " ", r.text)

def make_extraction_prompt(article: dict) -> str:
    return f"""
Extract ONLY explicit antimicrobial susceptibility/MIC observations from this article.
Return valid JSON with key "observations", an array of objects with:
compound, organism, strain, method, mic_value, mic_unit, mic_operator,
endpoint, source_location, evidence_quote.
Do not infer missing values. Preserve inequality operators (>, >=, <, <=).
Use null for unknown fields. One observation per compound-organism-strain-condition.
If no MIC data are present, return {{ "observations": [] }}.
Article title: {article.get('title','')}
PMID: {article.get('pmid','')}
Abstract/full text:
{article.get('text', article.get('abstract',''))[:18000]}
"""

def call_llm(prompt: str) -> str:
    provider = os.getenv("LLM_PROVIDER", "ollama").lower()
    if provider == "openai":
        from openai import OpenAI
        client = OpenAI()
        model = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
        response = client.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":"You extract structured scientific data conservatively. Return JSON only."},
                      {"role":"user","content":prompt}],
            response_format={"type":"json_object"},
            temperature=0
        )
        return response.choices[0].message.content
    import ollama
    model = os.getenv("OLLAMA_MODEL", "llama3.1")
    response = ollama.chat(model=model, messages=[
        {"role":"system","content":"You extract structured scientific data conservatively. Return JSON only."},
        {"role":"user","content":prompt}
    ], format="json", options={"temperature":0})
    return response["message"]["content"]

def extract_candidates(records_csv: str | Path | None = None, use_fulltext: bool = True) -> pd.DataFrame:
    path = Path(records_csv) if records_csv else OUT / "pubmed_records.csv"
    articles = pd.read_csv(path).fillna("")
    all_obs = []
    for i, art in articles.iterrows():
        text = art["abstract"]
        if use_fulltext and art.get("pmcid"):
            ft = fetch_europe_pmc_fulltext(art["pmcid"])
            if ft: text = ft
        article = {"title":art["title"], "pmid":art["pmid"], "text":text}
        try:
            raw = call_llm(make_extraction_prompt(article))
            data = json.loads(raw)
            for obs in data.get("observations", []):
                obs["pmid"] = art["pmid"]
                obs["title"] = art["title"]
                obs["pubmed_url"] = art["pubmed_url"]
                obs["verification_status"] = "needs_human_verification"
                all_obs.append(obs)
        except Exception as e:
            print(f"Extraction failed for PMID {art['pmid']}: {e}")
    df = pd.DataFrame(all_obs)
    if not df.empty:
        df.to_csv(OUT / "mic_candidates_unverified.csv", index=False)
    return df

def normalize_mic(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize common mass-concentration units to mg/L; retain censoring operator."""
    out = df.copy()
    if "mic_value" not in out: return out
    out["mic_value_numeric"] = pd.to_numeric(out["mic_value"], errors="coerce")
    units = out.get("mic_unit", pd.Series("", index=out.index)).astype(str).str.lower().str.replace("µ","u")
    factors = {"mg/l":1.0, "ug/ml":1.0, "μg/ml":1.0, "mcg/ml":1.0,
               "mg/ml":1000.0, "ug/l":0.001, "mcg/l":0.001}
    out["mic_mg_l"] = [v * factors.get(u.replace(" ",""), float("nan"))
                       if pd.notna(v) else float("nan")
                       for v,u in zip(out["mic_value_numeric"], units)]
    return out

def main():
    p = argparse.ArgumentParser()
    p.add_argument("command", choices=["retrieve","extract","normalize"])
    p.add_argument("--retmax", type=int, default=200)
    p.add_argument("--no-fulltext", action="store_true")
    args = p.parse_args()
    if args.command == "retrieve":
        retrieve_literature(args.retmax)
    elif args.command == "extract":
        extract_candidates(use_fulltext=not args.no_fulltext)
    elif args.command == "normalize":
        src = OUT / "mic_candidates_unverified.csv"
        df = normalize_mic(pd.read_csv(src))
        df.to_csv(OUT / "mic_candidates_normalized_unverified.csv", index=False)
        print(f"Saved {len(df)} candidate rows")

if __name__ == "__main__":
    main()
'''

files["molecular_descriptors.py"] = r'''
"""Retrieve candidate structures from PubChem and calculate RDKit descriptors."""
from pathlib import Path
import time
import pandas as pd
import requests

ROOT = Path(__file__).resolve().parent
COMPOUNDS = pd.read_csv(ROOT / "data" / "cephalosporins_20.csv")
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

def pubchem_smiles(name):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/CanonicalSMILES,IsomericSMILES/JSON"
    r = requests.get(url, timeout=45)
    if r.status_code != 200: return None
    props = r.json().get("PropertyTable", {}).get("Properties", [])
    if not props: return None
    return props[0].get("SMILES") or props[0].get("ConnectivitySMILES")

def main():
    rows=[]
    for r in COMPOUNDS.itertuples(index=False):
        try: smi = pubchem_smiles(r.pubmed_name)
        except Exception as e:
            print(r.pubmed_name, e); smi=None
        rows.append({"compound":r.common_name,"pubchem_query":r.pubmed_name,"smiles":smi})
        time.sleep(0.25)
    df=pd.DataFrame(rows)
    df.to_csv(OUT/"compound_smiles.csv",index=False)
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
        desc=[]
        for row in df.itertuples(index=False):
            mol=Chem.MolFromSmiles(row.smiles) if isinstance(row.smiles,str) and row.smiles else None
            if mol is None:
                desc.append({"compound":row.compound,"valid_smiles":False})
                continue
            desc.append({
                "compound":row.compound,"valid_smiles":True,
                "MW":Descriptors.MolWt(mol),"LogP":Crippen.MolLogP(mol),
                "TPSA":rdMolDescriptors.CalcTPSA(mol),
                "HBD":rdMolDescriptors.CalcNumHBD(mol),
                "HBA":rdMolDescriptors.CalcNumHBA(mol),
                "RotatableBonds":rdMolDescriptors.CalcNumRotatableBonds(mol),
                "AromaticRings":rdMolDescriptors.CalcNumAromaticRings(mol)
            })
        pd.DataFrame(desc).to_csv(OUT/"molecular_descriptors.csv",index=False)
    except ImportError:
        print("RDKit not installed; SMILES saved, descriptors skipped.")
    print(df[["compound","smiles"]].to_string(index=False))

if __name__=="__main__": main()
'''

files["prepare_dataset.py"] = r'''
"""
Prepare human-verified MIC rows for analysis.
Expected input: output/mic_candidates_normalized_unverified.csv
Create a copy named output/mic_verified.csv after manual source checking.
"""
from pathlib import Path
import pandas as pd

ROOT=Path(__file__).resolve().parent
OUT=ROOT/"output"

def main():
    p=OUT/"mic_verified.csv"
    if not p.exists():
        raise SystemExit(
            "Create output/mic_verified.csv from the candidate file only after checking each row "
            "against the article. Do not train on unverified LLM output."
        )
    df=pd.read_csv(p)
    required={"compound","organism","mic_mg_l"}
    missing=required-set(df.columns)
    if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
    df["mic_mg_l"]=pd.to_numeric(df["mic_mg_l"],errors="coerce")
    df=df[df["mic_mg_l"].notna() & (df["mic_mg_l"]>0)].copy()
    print("Verified MIC observations:",len(df))
    print("Distinct compounds:",df["compound"].nunique())
    print("Distinct organisms:",df["organism"].nunique())
    print("Compounds represented:",", ".join(sorted(df["compound"].unique())))
    df.to_csv(OUT/"analysis_dataset.csv",index=False)

if __name__=="__main__": main()
'''

files["analysis_ml.py"] = r'''
"""
Exploratory MIC modelling.
This is a baseline panel model, NOT a validated QSAR. It requires human-verified MIC data.
GroupKFold keeps all observations for a compound in one fold to reduce leakage.
"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

ROOT=Path(__file__).resolve().parent
OUT=ROOT/"output"

def main():
    path=OUT/"analysis_dataset.csv"
    if not path.exists(): raise SystemExit("Run prepare_dataset.py first.")
    df=pd.read_csv(path)
    if "compound" not in df or "organism" not in df or "mic_mg_l" not in df:
        raise ValueError("Dataset must contain compound, organism, mic_mg_l.")
    # A MIC panel can have many rows but only 20 unique compounds.
    n_compounds=df["compound"].nunique()
    print(f"MIC rows: {len(df)} | distinct compounds: {n_compounds} | organisms: {df['organism'].nunique()}")
    if n_compounds < 5:
        raise ValueError("Too few distinct compounds for even exploratory grouped validation (minimum set here: 5).")
    df=df[df["mic_mg_l"]>0].copy()
    df["log10_mic"]=np.log10(df["mic_mg_l"])
    features=["compound","organism"]
    X=df[features]; y=df["log10_mic"]; groups=df["compound"]
    pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore"),features)])
    models={
        "RandomForest":RandomForestRegressor(n_estimators=400,min_samples_leaf=2,random_state=42),
        "ExtraTrees":ExtraTreesRegressor(n_estimators=400,min_samples_leaf=2,random_state=42),
        "HistGradientBoosting":HistGradientBoostingRegressor(max_iter=150,l2_regularization=1.0,random_state=42)
    }
    n_splits=min(5,n_compounds)
    cv=GroupKFold(n_splits=n_splits)
    rows=[]
    for name,model in models.items():
        pipe=make_pipeline(pre,model)
        scores=cross_validate(pipe,X,y,groups=groups,cv=cv,
            scoring={"mae":"neg_mean_absolute_error","rmse":"neg_root_mean_squared_error","r2":"r2"},
            error_score="raise")
        rows.append({"model":name,"MAE_log10":-scores["test_mae"].mean(),
                     "RMSE_log10":-scores["test_rmse"].mean(),"R2":scores["test_r2"].mean()})
    result=pd.DataFrame(rows).sort_values("MAE_log10")
    result.to_csv(OUT/"model_comparison.csv",index=False)
    print(result.to_string(index=False))
    print("Interpret cautiously: only compound/organism categories are used; no molecular descriptors or strain/method covariates.")

if __name__=="__main__": main()
'''

files["R/01_read_and_summarise.R"] = r'''
# Run from the project root in RStudio
library(readr)
library(dplyr)
library(ggplot2)

file <- "output/analysis_dataset.csv"
if (!file.exists(file)) stop("Run the Python retrieval/extraction and verification steps first.")

mic <- read_csv(file, show_col_types = FALSE) |>
  mutate(mic_mg_l = as.numeric(mic_mg_l),
         log10_mic = log10(mic_mg_l)) |>
  filter(is.finite(log10_mic), mic_mg_l > 0)

cat("MIC observations:", nrow(mic), "\n")
cat("Distinct cephalosporins:", n_distinct(mic$compound), "\n")
cat("Distinct organisms:", n_distinct(mic$organism), "\n")

summary_table <- mic |>
  group_by(compound) |>
  summarise(n_observations = n(),
            n_organisms = n_distinct(organism),
            median_MIC_mg_L = median(mic_mg_l, na.rm = TRUE),
            min_MIC_mg_L = min(mic_mg_l, na.rm = TRUE),
            max_MIC_mg_L = max(mic_mg_l, na.rm = TRUE),
            .groups = "drop") |>
  arrange(median_MIC_mg_L)

print(summary_table)
write_csv(summary_table, "output/R_compound_summary.csv")

p <- ggplot(mic, aes(x = reorder(compound, mic_mg_l, FUN = median),
                     y = mic_mg_l)) +
  geom_boxplot(outlier.alpha = 0.35) +
  scale_y_log10() +
  coord_flip() +
  labs(x = "Cephalosporin", y = "MIC (mg/L, log scale)",
       title = "Verified MIC observations by cephalosporin") +
  theme_minimal()
ggsave("output/MIC_by_cephalosporin.png", p, width = 10, height = 8, dpi = 300)
'''

files["README.md"] = r'''
# Cephalosporin antimicrobial activity & MIC pipeline (20 compounds)

This project retrieves PubMed records, attempts full-text retrieval through Europe PMC,
uses Ollama or OpenAI to propose structured MIC extractions, retrieves PubChem structures,
and provides R/Python exploratory analysis.

## Included cephalosporins
Cefalexin, cefadroxil, cefradine, cefazolin, cefuroxime, cefaclor, cefprozil,
cefoxitin, cefotetan, cefotiam, cefixime, cefpodoxime, cefdinir, ceftriaxone,
cefotaxime, ceftazidime, cefoperazone, cefepime, ceftaroline, ceftobiprole.

## 1. Install (Linux / JupyterLab / RStudio)
```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env
```
Set `NCBI_EMAIL`. For OpenAI, set `LLM_PROVIDER=openai` and `OPENAI_API_KEY`.
For local Ollama, start Ollama and pull a model, e.g. `ollama pull llama3.1`.

## 2. Retrieve PubMed literature
```bash
python cephalo_pipeline.py retrieve --retmax 200
```
The query is run separately for each of the 20 compounds, then duplicate PMIDs are removed.
`retmax` is per compound, not a guarantee that all relevant literature has been captured.

## 3. Extract MIC candidates with Ollama or OpenAI
```bash
python cephalo_pipeline.py extract
```
To use abstracts only:
```bash
python cephalo_pipeline.py extract --no-fulltext
```
Output: `output/mic_candidates_unverified.csv`.
This is an extraction aid, not a verified dataset. The model may omit rows, misread tables,
or incorrectly associate values. Every observation must be checked against the source.

## 4. Normalize units
```bash
python cephalo_pipeline.py normalize
```
Output: `output/mic_candidates_normalized_unverified.csv`.
Common mass concentration units are converted to mg/L. Censored MICs retain their operator
in a separate field. Do not silently convert censored values to exact values.

## 5. Retrieve structures and calculate descriptors
```bash
python molecular_descriptors.py
```
Outputs `output/compound_smiles.csv` and, if RDKit is installed,
`output/molecular_descriptors.csv`. Check stereochemistry and salt/tautomer identity.

## 6. Human verification (required)
Review each candidate row against the article. Record the exact organism/strain, method,
medium, incubation conditions, endpoint, MIC operator/value/unit, and source table/page.
Correct the normalized value and mark `verification_status=verified`.
Copy the checked file to `output/mic_verified.csv`, retaining at least:
`compound, organism, mic_mg_l`.
Recommended additional columns: `strain, method, mic_operator, endpoint, pmid,
source_location, evidence_quote, verification_status`.

Then run:
```bash
python prepare_dataset.py
python analysis_ml.py
```

## 7. R summary and plot
Open `R/01_read_and_summarise.R` in RStudio with working directory set to the project root.

## Important scientific limitations
- PubMed abstracts rarely contain complete MIC matrices. Full-text access is incomplete.
- Search results are literature candidates, not a systematic-review-complete corpus.
- MIC values depend on organism, strain, test method, medium, inoculum, incubation, and breakpoint standards.
- MICs from different organisms or methods should not be pooled without modelling these factors.
- Repeated organism/strain observations do not create new distinct compounds.
- The supplied ML script is a small exploratory compound/organism categorical baseline, not a robust QSAR.
  With only 20 molecules, use grouped validation by compound and report uncertainty.
- Do not interpret lower MIC values across unrelated organisms as a simple ranking of antibiotic potency.
- Check current antimicrobial susceptibility standards (e.g., CLSI/EUCAST) separately for clinical interpretation.

## Suggested research-grade extensions
1. Add a PRISMA-style screening log and deduplication.
2. Include only standardized broth microdilution results for a primary analysis.
3. Keep separate datasets by organism group and test method.
4. Encode censored MIC data using interval/censored modelling rather than treating inequalities as exact.
5. Add RDKit descriptors only after structure identity and stereochemistry are validated.
6. Compare models with leave-one-compound-out validation; do not split individual MIC rows randomly.
'''

files["notebooks/Cephalosporin_20_pipeline.ipynb"] = json.dumps({
 "cells":[
  {"cell_type":"markdown","metadata":{},"source":["# Cephalosporin MIC literature pipeline — 20 compounds\n","Run cells from the project root. Verify all AI-extracted values against the source paper."]},
  {"cell_type":"code","execution_count":None,"metadata":{},"outputs":[],"source":["%pip install -r requirements.txt"]},
  {"cell_type":"code","execution_count":None,"metadata":{},"outputs":[],"source":["from cephalo_pipeline import retrieve_literature\n","records = retrieve_literature(retmax=200)\n","records.head()"]},
  {"cell_type":"markdown","metadata":{},"source":["## Next steps\n","Run `python cephalo_pipeline.py extract`, review candidate MIC rows manually, then run `normalize`, `prepare_dataset.py`, and `analysis_ml.py`."]},
  {"cell_type":"code","execution_count":None,"metadata":{},"outputs":[],"source":["import pandas as pd\n","compounds = pd.read_csv('data/cephalosporins_20.csv')\n","compounds"]}],
 "metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python"}},
 "nbformat":4,"nbformat_minor":5
}, indent=2)

for rel, content in files.items():
    p = root / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")

# Compile Python scripts to catch syntax errors
for name in ["cephalo_pipeline.py", "molecular_descriptors.py", "prepare_dataset.py", "analysis_ml.py"]:
    py_compile.compile(str(root/name), doraise=True)

zip_path = Path("/mnt/data/cephalosporin_20_pipeline.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob("*"):
        if p.is_file() and "__pycache__" not in str(p):
            z.write(p, p.relative_to(root.parent))
print("Created:", zip_path, "size:", zip_path.stat().st_size)
